<!--
(C) Copyright 1996- ECMWF.

This software is licensed under the terms of the Apache Licence Version 2.0
which can be obtained at http://www.apache.org/licenses/LICENSE-2.0.
In applying this licence, ECMWF does not waive the privileges and immunities
granted to it by virtue of its status as an intergovernmental organisation
nor does it submit to any jurisdiction.
-->

> **Modified copy.** One Weather Lab adapted this notebook from `examples/thermofeel-earthkit-demo.ipynb` in [ecmwf/thermofeel](https://github.com/ecmwf/thermofeel), release 2.3.0 (Apache-2.0, &copy; ECMWF). The changes include (i) the apparent temperature, computed with the Steadman (1984) formula in place of thermofeel's own apparent temperature function; (ii) the added simplified wet-bulb globe temperature (WBGT), net effective temperature and wind chill temperature, with their maps; (iii) summary statistics for Greece or Europe in place of the global domain; (iv) the added WBGT by the Liljegren method, computed from open-data pressure and solar radiation; (v) maps of the humidex, the simplified WBGT, the Liljegren WBGT and the wind chill temperature by class; (vi) links to the companion scripts that point to the upstream release; (vii) a Binder badge.

**Run the tutorial via Binder:**

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/one-weather-lab/biometeorology-air-quality-and-human-health/HEAD?labpath=notebooks%2Fdirect_thermal_indices.ipynb)

# thermofeel &times; earthkit &mdash; thermal-comfort indices from ECMWF open data

[**thermofeel**](https://github.com/ecmwf/thermofeel) is a pure-`numpy` library
that computes human **thermal-comfort indices** (Universal Thermal Climate Index,
apparent temperature, humidex, heat index, wet-bulb globe temperature, wind
chill, &hellip;) from meteorological fields.

This notebook is a small, self-contained demo built entirely on the
[**earthkit**](https://earthkit.readthedocs.io) stack:

- **earthkit-data** pulls a recent forecast from the **ECMWF open data** service
  &mdash; no login or API key required (data is CC-BY-4.0);
- **thermofeel** + **earthkit-meteo** turn the raw fields into comfort indices;
- **earthkit-plots** draws the maps &mdash; this replaces the old
  Magics / metview / seaborn plotting code.

### What open data can (and cannot) do

The free open-data feed ships the surface analysis/forecast fields but **not** the
radiation components &mdash; notably the direct solar radiation `fdir` &mdash;
that the radiation-based indices need. So this demo sticks to the
**humidity/wind indices** (apparent temperature, heat index, humidex, discomfort
index, net effective temperature, wind chill temperature and simplified WBGT),
which need only `2t`, `2d`, `10u` and `10v`.

The radiation indices (**UTCI**, **mean radiant temperature**, WBGT) require an
authenticated, radiation-complete source. The companion script
[`examples/compute-thermal-indices.py`](https://github.com/ecmwf/thermofeel/blob/2.3.0/examples/compute-thermal-indices.py) is built for
that &mdash; see its `--source polytope` / `--source mars` options and the
`--approximate-fdir` fallback &mdash; and
[`examples/compute-obs.py`](https://github.com/ecmwf/thermofeel/blob/2.3.0/examples/compute-obs.py) computes indices from station
observations. Pointers to both are collected at the end.

In [ ]:
# Core scientific stack
import numpy as np

# earthkit: data access, meteorological helpers and plotting
import earthkit.data as ekd
import earthkit.meteo as ekm
from earthkit.meteo import wind

import earthkit.plots as ekp

# thermofeel: the thermal-comfort indices themselves
import thermofeel as tf

# requests: only used for the optional METAR station overlay near the end
import requests

# matplotlib is earthkit-plots' rendering backend; we only touch it to display
# each chart deterministically when the notebook is run head-less (nbconvert).
import io

import matplotlib.pyplot as plt
from IPython.display import Image, display


def show(chart, dpi=85):
    '''Render an earthkit-plots chart to an inline PNG, then close its figure.

    Rendering explicitly - rather than leaning on matplotlib's implicit
    end-of-cell display - keeps `jupyter nbconvert` output reproducible (it also
    works for cells that add a scatter overlay) and keeps the embedded images
    light. The crisp copies are still written to disk by `chart.save(...)`.
    '''
    buffer = io.BytesIO()
    chart.fig.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
    plt.close(chart.fig)
    display(Image(data=buffer.getvalue()))


print("thermofeel    ", tf.__version__)
print("earthkit-data ", ekd.__version__)
print("earthkit-meteo", ekm.__version__)
print("earthkit-plots", ekp.__version__)
print("numpy         ", np.__version__)

## 1. Fetch a recent forecast from ECMWF open data

`earthkit.data.from_source("ecmwf-open-data", ...)` wraps the
[`ecmwf-opendata`](https://github.com/ecmwf/ecmwf-opendata) client, which always
resolves to the **latest available HRES run** &mdash; so we never hard-code a
date. We request the four surface fields the humidity/wind indices depend on:

| shortName | meaning            | units |
|-----------|--------------------|-------|
| `2t`      | 2&nbsp;m temperature | K     |
| `2d`      | 2&nbsp;m dewpoint    | K     |
| `10u`     | 10&nbsp;m U-wind     | m/s   |
| `10v`     | 10&nbsp;m V-wind     | m/s   |

In [ ]:
# Live or retrospective. None reads the latest run from the ECMWF server. A date in YYYYMMDD form
# reads that run from the AWS mirror instead, at the cycle hour RUN_TIME. The 0 UTC run with step 12
# is valid at 12 UTC, near the afternoon heat peak. The first usable 0 UTC run is 6 March 2024
# ("20240306"), since runs before it lack the 0.25 degree grid or fields the notebook reads.
RUN_DATE = "20260623"  # None = live; "YYYYMMDD" = retrospective
RUN_TIME = 0           # [UTC hour] used only when RUN_DATE is set

request = dict(
    type="fc",
    stream="oper",
    levtype="sfc",
    step=12,
    param=["2t", "2d", "10u", "10v"],
)
if RUN_DATE is None:
    DATA_SOURCE = "ecmwf"
else:
    DATA_SOURCE = "aws"
    request.update(date=RUN_DATE, time=RUN_TIME)

# Exponential backoff (starting at 5s up to 60s, max 10 tries) so transient
# cloud mirror throttling (HTTP 503) recovers quickly without default 120s stalls.
ds = ekd.from_source(
    "ecmwf-open-data",
    source=DATA_SOURCE,
    request=request,
    retry_after=(5, 60, 2),
    maximum_retries=10,
)
fl = ds.to_fieldlist()

# This build of earthkit-data doesn't carry a working `.sel()` for open-data
# GRIB, so we index the fields by their GRIB shortName instead.
fields = {f.metadata("shortName"): f for f in fl}

# Field values -> 2-D numpy arrays (Kelvin for 2t/2d, m/s for the winds).
t2m = fields["2t"].to_numpy()    # [K]
d2m = fields["2d"].to_numpy()    # [K]
u10 = fields["10u"].to_numpy()   # [m/s]
v10 = fields["10v"].to_numpy()   # [m/s]

# Coordinates: earthkit-data 1.0 exposes lat/lon through `field.geography`
# (the older `field.to_latlon()` is gone); `.latlons()` returns 2-D meshgrids.
lat2d, lon2d = fields["2t"].geography.latlons()

base = fields["2t"].metadata("base_datetime")
valid = fields["2t"].metadata("valid_datetime")
print(f"run       : {base}")
print(f"valid     : {valid}  (step {fields['2t'].metadata('step')} h)")
print(f"grid      : {t2m.shape}  (global regular lat/lon, 0.25 deg)")
print(f"lat range : {lat2d.min():6.2f} ... {lat2d.max():6.2f}")
print(f"lon range : {lon2d.min():6.2f} ... {lon2d.max():6.2f}")

## 2. Compute thermal-comfort indices

thermofeel works directly on the gridded `numpy` arrays and expects temperatures
in **Kelvin**. We first derive the two quantities several indices share:

- **wind speed** from the U/V components, via `earthkit.meteo.wind.speed`;
- **relative humidity** from temperature + dewpoint, via
  `thermofeel.calculate_relative_humidity_percent`.

Then we compute seven indices for variety: **apparent temperature**
(Steadman, 1984, in the shade; needs `2t` + wind + RH), **heat index (adjusted)**,
**humidex**, **Thom's discomfort index**, the **simplified wet-bulb globe
temperature** (WBGT, needs `2t` + RH), the **net effective temperature** (needs
`2t` + wind + RH) and the **wind chill temperature** (needs `2t` + wind). The
thermofeel functions return Kelvin, which we convert to &deg;C for display, and the
apparent temperature is computed directly in &deg;C. Most of these are
heat-stress indices, so their values are most meaningful in warm conditions, while
the wind chill temperature applies to cold and windy conditions. The summary
statistics cover Greece at the valid time of the forecast field, with Europe
available as an alternative region.

In [ ]:
# Shared inputs.
ws = wind.speed(u10, v10)                                        # [m/s]
rh = tf.calculate_relative_humidity_percent(t2_k=t2m, td_k=d2m)  # [%]


def steadman_apparent_temperature(t2_k, rh, va):
    """Steadman (1984) apparent temperature in the shade.

    Parameters
    ----------
    t2_k : array-like
        2 m temperature [K].
    rh : array-like
        Relative humidity [%].
    va : array-like
        Wind speed at 10 m [m/s].

    Returns
    -------
    numpy.ndarray
        Apparent temperature [C].
    """
    vapour_pressure_kpa = tf.calculate_nonsaturation_vapour_pressure(t2_k, rh) / 10.0  # [hPa] -> [kPa]
    return -2.7 + 1.04 * tf.kelvin_to_celsius(t2_k) + 2.0 * vapour_pressure_kpa - 0.65 * va


# Indices (thermofeel returns Kelvin -> convert to Celsius for display; the
# apparent temperature is already in Celsius).
to_c = tf.kelvin_to_celsius
apparent_c   = steadman_apparent_temperature(t2_k=t2m, rh=rh, va=ws)
heatindex_c  = to_c(tf.calculate_heat_index_adjusted(t2_k=t2m, td_k=d2m))
humidex_c    = to_c(tf.calculate_humidex(t2_k=t2m, td_k=d2m))
discomfort_c = to_c(tf.calculate_discomfort_index(t2_k=t2m, rh=rh))
wbgt_c       = to_c(tf.calculate_wbgt_simple(t2_k=t2m, rh=rh))
net_c        = to_c(tf.calculate_normal_effective_temperature(t2_k=t2m, va=ws, rh=rh))

# The wind chill formula is valid for -50 to 5 C and 5 to 80 km/h, so all other cells are NaN.
WIND_CHILL_T2_RANGE_K = (223.15, 278.15)   # [K] -50 to 5 C
WIND_CHILL_SPEED_RANGE_KMH = (5.0, 80.0)   # [km/h]
MS_TO_KMH = 3.6
ws_kmh = ws * MS_TO_KMH
wind_chill_valid = (
    (t2m >= WIND_CHILL_T2_RANGE_K[0]) & (t2m <= WIND_CHILL_T2_RANGE_K[1])
    & (ws_kmh >= WIND_CHILL_SPEED_RANGE_KMH[0]) & (ws_kmh <= WIND_CHILL_SPEED_RANGE_KMH[1])
)
windchill_c  = np.where(wind_chill_valid, to_c(tf.calculate_wind_chill(t2_k=t2m, va=ws)), np.nan)

# Statistics region as (south, north, west, east) in degrees. The rectangles are
# approximate and include sea and neighboring land.
STATS_REGIONS = {
    "Greece": (34.5, 42.0, 19.0, 29.5),
    "Europe": (34.0, 72.0, -25.0, 45.0),
}
STATS_REGION = "Greece"
south, north, west, east = STATS_REGIONS[STATS_REGION]
region_mask = (lat2d >= south) & (lat2d <= north) & (lon2d >= west) & (lon2d <= east)

print(f"Statistics for {STATS_REGION} ({south} to {north} N, {west} to {east} E), valid {valid} UTC")
print(f"{'field':32s} {'min':>8s} {'mean':>8s} {'max':>8s}")
print("-" * 58)
for name, arr in [
    ("2 m temperature [C]", to_c(t2m)),
    ("relative humidity [%]", rh),
    ("wind speed [m/s]", ws),
    ("apparent temperature [C]", apparent_c),
    ("heat index adjusted [C]", heatindex_c),
    ("humidex [C]", humidex_c),
    ("discomfort index [C]", discomfort_c),
    ("simplified WBGT [C]", wbgt_c),
    ("net effective temperature [C]", net_c),
    ("wind chill temperature [C]", windchill_c),
]:
    region_values = arr[region_mask]
    if np.isfinite(region_values).any():
        print(f"{name:32s} {np.nanmin(region_values):8.1f} {np.nanmean(region_values):8.1f} {np.nanmax(region_values):8.1f}")
    else:
        print(f"{name:32s} {'n/a':>8s} {'n/a':>8s} {'n/a':>8s}")

## 2.1 Radiation-based wet-bulb globe temperature

The simplified WBGT needs only the 2 m temperature and the relative humidity. The Liljegren method solves the energy balance of the natural wet bulb and of the black globe, so it also needs the surface pressure, the wind speed and the solar radiation. Pressure and radiation are fetched from the same forecast run. The solar radiation is accumulated from the start of the forecast, so the difference between steps 9 and 12 gives the mean flux over the 3 h before the valid time, and the cosine of the solar zenith angle is averaged over the same window. Open data does not include the direct solar radiation, so its share of the downward radiation is estimated from the downward radiation and the zenith angle (`--approximate-fdir`). The estimate is approximate, so the map of this index shows indicative values.

In [ ]:
from datetime import datetime, timedelta

from earthkit.meteo import solar
from thermofeel.approximations import approximate_fdir_erbs

# Surface pressure and downward solar radiation of the same run. The radiation is accumulated since
# the start of the forecast, so the difference between two steps is the mean flux of the window
# that ends at the valid time.
WINDOW_H = 3
step_h = request["step"]
radiation = ekd.from_source(
    "ecmwf-open-data",
    source=DATA_SOURCE,
    request=dict(
        type="fc",
        stream="oper",
        levtype="sfc",
        param=["sp", "ssrd"],
        date=str(base)[:10].replace("-", ""),
        time=int(str(base)[11:13]),
        step=[step_h - WINDOW_H, step_h],
    ),
    retry_after=(5, 60, 2),
    maximum_retries=10,
).to_fieldlist()
rad = {(f.metadata("shortName"), int(f.metadata("step"))): f.to_numpy() for f in radiation}
ssrd_mean = np.maximum((rad["ssrd", step_h] - rad["ssrd", step_h - WINDOW_H]) / (WINDOW_H * 3600.0), 0.0)  # [W/m2]
sp_hpa = rad["sp", step_h] / 100.0  # [hPa]

# The solver is iterative, so it runs on a box around Europe that covers the maps and the statistics region.
box = (lat2d >= 20.0) & (lat2d <= 75.0) & (lon2d >= -30.0) & (lon2d <= 60.0)
valid_time = datetime.fromisoformat(str(valid))
window_start = valid_time - timedelta(hours=WINDOW_H)
cossza = solar.cos_solar_zenith_angle_integrated(window_start, valid_time, lat2d[box], lon2d[box])  # window mean
day_of_year = valid_time.timetuple().tm_yday

# Open data has no direct solar radiation, so its share of the downward radiation is estimated from
# the downward radiation and the solar zenith angle.
fdir_flux = approximate_fdir_erbs(ssrd_mean[box], cossza, doy=day_of_year)  # [W/m2]
direct_fraction = np.divide(fdir_flux, ssrd_mean[box], out=np.zeros_like(fdir_flux), where=ssrd_mean[box] > 0)

wbgt_liljegren_c = np.full(t2m.shape, np.nan)
wbgt_liljegren_c[box] = to_c(
    tf.calculate_wbgt_liljegren(
        t2_k=t2m[box],
        rh=rh[box],
        pressure=sp_hpa[box],
        va=ws[box],
        ssrd=ssrd_mean[box],
        fdir=direct_fraction,
        cossza=cossza,
    )
)

print(f"{'field':32s} {'min':>8s} {'mean':>8s} {'max':>8s}")
print("-" * 58)
for name, arr in [("simplified WBGT [C]", wbgt_c), ("Liljegren WBGT [C]", wbgt_liljegren_c)]:
    region_values = arr[region_mask]
    print(f"{name:32s} {np.nanmin(region_values):8.1f} {np.nanmean(region_values):8.1f} {np.nanmax(region_values):8.1f}")

## 3. Map the indices with earthkit-plots

`earthkit-plots` is the modern replacement for Magics. There are two entry
points:

- `ekp.quickplot(field, domain="Europe", units="celsius")` &mdash; a one-liner
  straight from an earthkit-data *field*, ideal for a quick look at an input;
- `ekp.Map(domain=...)` + `.contourf(...)` &mdash; full control, and the natural
  fit for our derived indices, which are plain `numpy` arrays. We hand the array
  plus the `lon2d`/`lat2d` meshgrids to `.contourf`, and describe the values with
  `metadata={"units": "celsius", ...}` so the colour bar is labelled correctly.

`domain="Europe"` clips the global field to Europe; `coastlines`, `borders`,
`gridlines`, `legend` and `title` add the usual map furniture. Figures are also
saved as PNGs next to the notebook (git-ignored).

The humidex, the simplified WBGT and the wind chill temperature are mapped by
class. Cells below the lowest class are left blank. The Liljegren WBGT is mapped
with the same classes as the simplified WBGT, directly below it, so the two can
be compared. The apparent temperature and the net effective temperature have no
universal scale and keep a continuous colour scale.

Finally, as an **optional** extra, we pull a handful of recent **METARs** from
NOAA's Aviation Weather API (`aviationweather.gov`, no auth), compute the
apparent temperature per station with the same Steadman formula, and overlay
them on the model field with `chart.scatter(..., z=values)` &mdash; same colour
scale, so a well-matched station blends into the background. The fetch is
wrapped in `try/except` so a transient API hiccup can't break the notebook.

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# (1) One-liner straight from an earthkit-data field: the 2 m temperature input.
qp = ekp.quickplot(fields["2t"], domain="Europe", units="celsius")
qp.title("2 m temperature (model input)")
qp.save("t2m_input.png", dpi=150)
show(qp)


# (2) Full control: build a Map and contour an index (already in C).
def index_map(values_c, long_name, filename):
    chart = ekp.Map(domain="Europe")
    chart.contourf(
        values_c,
        x=lon2d,
        y=lat2d,
        metadata={"units": "celsius", "long_name": long_name},
    )
    chart.coastlines()
    chart.borders()
    chart.gridlines()
    chart.legend()
    chart.title(long_name)
    chart.save(filename, dpi=150)
    return chart


# (3) Classified maps: one colour per class of the index scale. The classes are
# (name, lower limit in C), and a class includes its lower limit. Wind chill classes are set by
# how far below zero the value lies, so its limits are applied to the negated values.
HUMIDEX_CLASSES = [("Little discomfort", 20), ("Some discomfort", 30), ("Great discomfort", 40), ("Dangerous", 46)]
HUMIDEX_COLORS = ["#f1ecf4", "#d4b0d3", "#d966a6", "#cc1f5a"]
WBGT_CLASSES = [
    ("Low risk", -np.inf), ("Moderate risk", 26.7), ("High risk", 29.5),
    ("Very high risk", 31.2), ("Extremely high risk", 32.3),
]
WBGT_COLORS = ["#fdf3c8", "#fbd36b", "#f39c3d", "#d2492f", "#7a1a1a"]
WIND_CHILL_CLASSES = [
    ("Low risk", 0), ("Moderate risk", 10), ("High risk", 28),
    ("Very high risk", 40), ("Severe risk", 48), ("Extreme risk", 55),
]
WIND_CHILL_COLORS = ["#e3f1fa", "#a8d0ea", "#6a9fd6", "#4a6fc0", "#5c3d99", "#3a1d66"]


def class_map(values_c, classes, colors, long_name, filename, sign=1):
    """Map an index by class, leaving cells below the lowest class blank.

    Parameters
    ----------
    values_c : numpy.ndarray
        Index values [C] on the lon2d/lat2d grid.
    classes : list of tuple
        Class name and lower limit, ordered from the lowest class to the highest.
    colors : list of str
        One colour per class.
    long_name : str
        Index name used in the title.
    filename : str
        Name of the saved PNG.
    sign : int
        1 where classes rise with the value, -1 where they rise as the value falls.

    Returns
    -------
    earthkit.plots.Map or None
        The map, or None when no cell falls in any class.
    """
    limits = [limit for _, limit in classes]
    class_index = np.digitize(sign * values_c, limits)  # 0 below the lowest class
    class_index[~np.isfinite(values_c)] = 0
    if not class_index.any():
        print(f"{long_name} map skipped (no cell falls in a class)")
        return None

    chart = ekp.Map(domain="Europe")
    for number, color in enumerate(colors, start=1):
        in_class = (class_index == number).astype(float)
        if in_class.any():
            # The 0.5 contour of the 0/1 field lies midway between cells inside and outside the class.
            chart.contourf(
                in_class,
                x=lon2d,
                y=lat2d,
                style=ekp.styles.Style(colors=ListedColormap([color]), levels=[0.5, 1.5], legend_style=None),
            )
    chart.coastlines()
    chart.borders()
    chart.gridlines()
    chart.title(f"{long_name} classes (°C)")

    labels = []
    for number, (name, limit) in enumerate(classes):
        if number + 1 < len(classes):
            upper = sign * classes[number + 1][1]
            span = f"< {upper:g}" if limit == -np.inf else f"{sign * limit:g} to {upper:g}"
            labels.append(f"{name} ({span})")
        else:
            labels.append(f"{name} ({'≥' if sign > 0 else '≤'} {sign * limit:g})")
    handles = [Patch(facecolor=c, edgecolor="#555555", linewidth=0.5, label=text) for c, text in zip(colors, labels)]
    chart.fig.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.0),
        ncols=min(len(handles), 2 if len(handles) == 4 else 3),
        frameon=False,
        fontsize=8,
    )
    chart.fig.savefig(filename, dpi=150, bbox_inches="tight")
    return chart


show(index_map(apparent_c, "Apparent temperature", "apparent_temperature.png"))
for chart in (
    class_map(humidex_c, HUMIDEX_CLASSES, HUMIDEX_COLORS, "Humidex", "humidex.png"),
    class_map(wbgt_c, WBGT_CLASSES, WBGT_COLORS, "Simplified WBGT", "wbgt_simplified.png"),
    class_map(wbgt_liljegren_c, WBGT_CLASSES, WBGT_COLORS, "Liljegren WBGT", "wbgt_liljegren.png"),
):
    if chart is not None:
        show(chart)
show(index_map(net_c, "Net effective temperature", "net_effective_temperature.png"))
chart = class_map(windchill_c, WIND_CHILL_CLASSES, WIND_CHILL_COLORS, "Wind chill temperature",
                  "wind_chill_temperature.png", sign=-1)
if chart is not None:
    show(chart)

In [ ]:
# Optional: overlay live METAR station observations on the model field. A retrospective run skips
# it, because the reports are current and would not match the model field.
STATIONS = "EGLL,LFPG,EDDF,EHAM,LEMD,LIRF,LSZH,LOWW,EKCH,ESSA,LEBL,EIDW,LPPT,LGAV"
url = f"https://aviationweather.gov/api/data/metar?ids={STATIONS}&format=json&hours=3"

s_lon, s_lat, s_at, s_id = [], [], [], []
if RUN_DATE is not None:
    print("METAR overlay skipped (retrospective run)")
else:
    try:
        reports = requests.get(url, timeout=30).json()
        latest = {}  # keep the most recent report per station
        for m in reports:
            if m.get("temp") is None or m.get("dewp") is None:
                continue
            k = m["icaoId"]
            if k not in latest or (m.get("obsTime") or 0) > latest[k].get("obsTime", 0):
                latest[k] = m
        for k, m in latest.items():
            t_k = tf.celsius_to_kelvin(float(m["temp"]))
            td_k = tf.celsius_to_kelvin(float(m["dewp"]))
            va = float(m.get("wspd") or 0.0) * 0.514444  # knots -> m/s
            rh_s = tf.calculate_relative_humidity_percent(t2_k=t_k, td_k=td_k)
            at_s = steadman_apparent_temperature(t2_k=t_k, rh=rh_s, va=va)
            s_lon.append(float(m["lon"]))
            s_lat.append(float(m["lat"]))
            s_at.append(float(at_s))
            s_id.append(k)
        print(f"{len(s_id)} stations: " + ", ".join(f"{i}={v:.0f}C" for i, v in zip(s_id, s_at)))
    except Exception as exc:  # keep the notebook runnable if the API is unreachable
        print(f"METAR fetch skipped ({exc!r})")

chart = ekp.Map(domain="Europe")
chart.contourf(
    apparent_c,
    x=lon2d,
    y=lat2d,
    metadata={"units": "celsius", "long_name": "Apparent temperature"},
)
if s_id:
    chart.scatter(
        x=np.array(s_lon),
        y=np.array(s_lat),
        z=np.array(s_at),
        metadata={"units": "celsius", "long_name": "Station apparent temperature"},
        s=90,
        edgecolors="black",
        linewidths=0.6,
    )
chart.coastlines()
chart.borders()
chart.gridlines()
chart.legend()
chart.title("Model apparent temperature + METAR stations" if s_id else "Model apparent temperature")
chart.save("apparent_temperature_stations.png", dpi=150)
show(chart)

## Where to go next

- **Radiation-based indices (UTCI, MRT, WBGT).** These need the surface radiation
  fields (`ssrd`, `ssr`, `fdir`, `strd`, `str`, &hellip;) that ECMWF open data
  doesn't publish. The example script
  [`compute-thermal-indices.py`](https://github.com/ecmwf/thermofeel/blob/2.3.0/examples/compute-thermal-indices.py) reads a
  radiation-complete source &mdash; see its `--source polytope` / `--source mars`
  options (install the optional `polytope-client` / `ecmwf-api-client` backends
  from the `thermofeel[examples]` extra), plus `--approximate-fdir` for when the
  direct radiation is missing &mdash; and encodes every index back to GRIB.
- **Station / observation workflows.**
  [`compute-obs.py`](https://github.com/ecmwf/thermofeel/blob/2.3.0/examples/compute-obs.py) computes indices (MRT, WBGT, &hellip;) from
  BSRN-style station observations, using `earthkit.meteo.solar` for the solar
  geometry.
- **The full index set.** See the
  [thermofeel docs](https://thermofeel.readthedocs.io) for UTCI, wet-bulb globe
  temperature (including the Liljegren method), wind chill, PMV/PPD, the KNMI heat
  force scale and the excess-heat / excess-cold factors.

_This notebook writes its maps as PNG files next to itself; those are git-ignored._